<a href="https://colab.research.google.com/github/alsoft69/3-6-final/blob/main/OilCalc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# ЯЧЕЙКА 1: Установка инструментов и сборка проекта
# ============================================================

# 1. Установка инструментов
!pip install buildozer cython

# 2. Системные зависимости
!apt-get update -y
!apt-get install -y openjdk-17-jdk python3-pip zip unzip autoconf libtool libffi-dev

# 3. Android SDK
!mkdir -p /opt/android-sdk/cmdline-tools
!wget -q https://dl.google.com/android/repository/commandlinetools-linux-9477386_latest.zip
!unzip -q -o commandlinetools-linux-9477386_latest.zip -d /opt/android-sdk/cmdline-tools
!mv /opt/android-sdk/cmdline-tools/cmdline-tools /opt/android-sdk/cmdline-tools/latest 2>/dev/null || true
!yes | /opt/android-sdk/cmdline-tools/latest/bin/sdkmanager --licenses 2>/dev/null || true
!/opt/android-sdk/cmdline-tools/latest/bin/sdkmanager "platforms;android-33" "build-tools;33.0.0" 2>/dev/null || true

# 4. NDK 25c
!mkdir -p /root/.buildozer/android/platform
%cd /root/.buildozer/android/platform
!wget -q https://dl.google.com/android/repository/android-ndk-r25c-linux.zip
!unzip -q -o android-ndk-r25c-linux.zip
!rm android-ndk-r25c-linux.zip
%cd /content/oil_app

# 5. Создание папки проекта
!mkdir -p /content/oil_app
%cd /content/oil_app

# 6. Запись main.py
main_py = r'''
import sqlite3
from datetime import datetime
import os

os.environ['KIVY_NO_CONSOLELOG'] = '1'

from kivy.app import App
from kivy.uix.boxlayout import BoxLayout
from kivy.uix.button import Button
from kivy.uix.label import Label
from kivy.uix.textinput import TextInput
from kivy.uix.spinner import Spinner
from kivy.uix.popup import Popup
from kivy.uix.scrollview import ScrollView
from kivy.uix.gridlayout import GridLayout
from kivy.uix.screenmanager import ScreenManager, Screen
from kivy.metrics import dp
from kivy.utils import platform

try:
    from plyer import share
    PLYER_AVAILABLE = True
except:
    PLYER_AVAILABLE = False

DB_NAME = None
DB_INITIALIZED = False

def get_db_path():
    global DB_NAME
    if DB_NAME is None:
        db_dir = os.path.join(os.environ.get('ANDROID_APP_PATH', os.path.expanduser('~')), '.oil_data')
        os.makedirs(db_dir, exist_ok=True)
        DB_NAME = os.path.join(db_dir, 'oil_accounting.db')
    return DB_NAME

def ensure_db():
    global DB_INITIALIZED
    if DB_INITIALIZED:
        return
    db_path = get_db_path()
    conn = sqlite3.connect(db_path)
    c = conn.cursor()
    c.execute("CREATE TABLE IF NOT EXISTS shops (id INTEGER PRIMARY KEY, name TEXT UNIQUE)")
    c.execute("CREATE TABLE IF NOT EXISTS machines (id INTEGER PRIMARY KEY, code TEXT UNIQUE, name TEXT, shop_id INTEGER)")
    c.execute("CREATE TABLE IF NOT EXISTS oils (id INTEGER PRIMARY KEY, name TEXT UNIQUE)")
    c.execute("CREATE TABLE IF NOT EXISTS records (id INTEGER PRIMARY KEY, date TEXT, machine_id INTEGER, oil_id INTEGER, liters REAL)")
    conn.commit()
    conn.close()
    DB_INITIALIZED = True
    return db_path

def get_shops():
    ensure_db()
    conn = sqlite3.connect(get_db_path())
    c = conn.cursor()
    c.execute("SELECT id, name FROM shops ORDER BY name")
    data = c.fetchall()
    conn.close()
    return data

def get_machines(shop_id=None):
    ensure_db()
    conn = sqlite3.connect(get_db_path())
    c = conn.cursor()
    if shop_id:
        c.execute("SELECT m.id, m.code, m.name FROM machines m WHERE m.shop_id=? ORDER BY m.code", (shop_id,))
    else:
        c.execute("SELECT m.id, m.code, m.name FROM machines m ORDER BY m.code")
    data = c.fetchall()
    conn.close()
    return data

def get_oils():
    ensure_db()
    conn = sqlite3.connect(get_db_path())
    c = conn.cursor()
    c.execute("SELECT id, name FROM oils ORDER BY name")
    data = c.fetchall()
    conn.close()
    return data

def add_shop(name):
    ensure_db()
    conn = sqlite3.connect(get_db_path())
    c = conn.cursor()
    try:
        c.execute("INSERT INTO shops (name) VALUES (?)", (name,))
        conn.commit()
        return True, "Цех добавлен"
    except sqlite3.IntegrityError:
        return False, "Такой цех уже есть"
    finally:
        conn.close()

def update_shop(shop_id, new_name):
    ensure_db()
    conn = sqlite3.connect(get_db_path())
    c = conn.cursor()
    try:
        c.execute("UPDATE shops SET name=? WHERE id=?", (new_name, shop_id))
        conn.commit()
        return True, "Цех обновлён"
    except sqlite3.IntegrityError:
        return False, "Цех с таким именем уже существует"
    finally:
        conn.close()

def delete_shop(shop_id):
    ensure_db()
    conn = sqlite3.connect(get_db_path())
    c = conn.cursor()
    c.execute("DELETE FROM machines WHERE shop_id=?", (shop_id,))
    c.execute("DELETE FROM shops WHERE id=?", (shop_id,))
    conn.commit()
    conn.close()

def add_machine(code, name, shop_id):
    ensure_db()
    conn = sqlite3.connect(get_db_path())
    c = conn.cursor()
    try:
        c.execute("INSERT INTO machines (code, name, shop_id) VALUES (?,?,?)", (code, name, shop_id))
        conn.commit()
        return True, "Станок добавлен"
    except sqlite3.IntegrityError:
        return False, "Такой код станка уже есть"
    finally:
        conn.close()

def update_machine(machine_id, new_code, new_name, new_shop_id):
    ensure_db()
    conn = sqlite3.connect(get_db_path())
    c = conn.cursor()
    try:
        c.execute("UPDATE machines SET code=?, name=?, shop_id=? WHERE id=?", (new_code, new_name, new_shop_id, machine_id))
        conn.commit()
        return True, "Станок обновлён"
    except sqlite3.IntegrityError:
        return False, "Код станка уже используется"
    finally:
        conn.close()

def delete_machine(machine_id):
    ensure_db()
    conn = sqlite3.connect(get_db_path())
    c = conn.cursor()
    c.execute("DELETE FROM records WHERE machine_id=?", (machine_id,))
    c.execute("DELETE FROM machines WHERE id=?", (machine_id,))
    conn.commit()
    conn.close()

def add_oil(name):
    ensure_db()
    conn = sqlite3.connect(get_db_path())
    c = conn.cursor()
    try:
        c.execute("INSERT INTO oils (name) VALUES (?)", (name,))
        conn.commit()
        return True, "Масло добавлено"
    except sqlite3.IntegrityError:
        return False, "Такое масло уже есть"
    finally:
        conn.close()

def update_oil(oil_id, new_name):
    ensure_db()
    conn = sqlite3.connect(get_db_path())
    c = conn.cursor()
    try:
        c.execute("UPDATE oils SET name=? WHERE id=?", (new_name, oil_id))
        conn.commit()
        return True, "Масло обновлено"
    except sqlite3.IntegrityError:
        return False, "Такое масло уже существует"
    finally:
        conn.close()

def delete_oil(oil_id):
    ensure_db()
    conn = sqlite3.connect(get_db_path())
    c = conn.cursor()
    c.execute("DELETE FROM records WHERE oil_id=?", (oil_id,))
    c.execute("DELETE FROM oils WHERE id=?", (oil_id,))
    conn.commit()
    conn.close()

def add_record(date, machine_id, oil_id, liters):
    ensure_db()
    conn = sqlite3.connect(get_db_path())
    c = conn.cursor()
    c.execute("INSERT INTO records (date, machine_id, oil_id, liters) VALUES (?,?,?,?)", (date, machine_id, oil_id, liters))
    conn.commit()
    conn.close()

def get_records():
    ensure_db()
    conn = sqlite3.connect(get_db_path())
    c = conn.cursor()
    c.execute("SELECT r.id, r.date, m.code, o.name, r.liters FROM records r JOIN machines m ON r.machine_id = m.id JOIN oils o ON r.oil_id = o.id ORDER BY r.date DESC, r.id DESC LIMIT 100")
    data = c.fetchall()
    conn.close()
    return data

def delete_record(rec_id):
    ensure_db()
    conn = sqlite3.connect(get_db_path())
    c = conn.cursor()
    c.execute("DELETE FROM records WHERE id=?", (rec_id,))
    conn.commit()
    conn.close()

def get_report(start_date, end_date):
    ensure_db()
    conn = sqlite3.connect(get_db_path())
    c = conn.cursor()
    c.execute("SELECT m.code, o.name, SUM(r.liters) FROM records r JOIN machines m ON r.machine_id = m.id JOIN oils o ON r.oil_id = o.id WHERE r.date >= ? AND r.date <= ? GROUP BY m.code, o.name ORDER BY m.code, o.name", (start_date, end_date))
    data = c.fetchall()
    conn.close()
    return data

class MainScreen(Screen):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        layout = BoxLayout(orientation='vertical', padding=dp(20), spacing=dp(15))
        layout.add_widget(Label(text='[b]УЧЕТ РАСХОДА МАСЛА[/b]', markup=True, font_size='20sp', size_hint_y=None, height=dp(60)))
        btn_add = Button(text='Добавить запись', size_hint=(0.8, None), height=dp(55), pos_hint={'center_x': 0.5})
        btn_add.bind(on_press=lambda x: setattr(self.manager, 'current', 'add_record'))
        layout.add_widget(btn_add)
        btn_journal = Button(text='Журнал записей', size_hint=(0.8, None), height=dp(55), pos_hint={'center_x': 0.5})
        btn_journal.bind(on_press=lambda x: setattr(self.manager, 'current', 'journal'))
        layout.add_widget(btn_journal)
        btn_report = Button(text='Отчет за месяц', size_hint=(0.8, None), height=dp(55), pos_hint={'center_x': 0.5})
        btn_report.bind(on_press=lambda x: setattr(self.manager, 'current', 'report'))
        layout.add_widget(btn_report)
        btn_ref = Button(text='Справочники', size_hint=(0.8, None), height=dp(55), pos_hint={'center_x': 0.5})
        btn_ref.bind(on_press=lambda x: setattr(self.manager, 'current', 'references'))
        layout.add_widget(btn_ref)
        self.add_widget(layout)

class AddRecordScreen(Screen):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.layout = BoxLayout(orientation='vertical', padding=dp(10), spacing=dp(8))
        header = BoxLayout(size_hint_y=None, height=dp(45))
        btn_back = Button(text='< Назад', size_hint_x=0.3)
        btn_back.bind(on_press=lambda x: setattr(self.manager, 'current', 'main'))
        header.add_widget(btn_back)
        header.add_widget(Label(text='Новая запись', size_hint_x=0.7))
        self.layout.add_widget(header)
        self.layout.add_widget(Label(text='Дата:', size_hint_y=None, height=dp(25)))
        self.date_input = TextInput(text=datetime.now().strftime('%Y-%m-%d'), size_hint_y=None, height=dp(40))
        self.layout.add_widget(self.date_input)
        self.layout.add_widget(Label(text='Цех:', size_hint_y=None, height=dp(25)))
        self.shop_spinner = Spinner(text='Выберите цех', size_hint_y=None, height=dp(44))
        self.shop_spinner.bind(text=self.on_shop_select)
        self.layout.add_widget(self.shop_spinner)
        self.layout.add_widget(Label(text='Станок:', size_hint_y=None, height=dp(25)))
        self.machine_spinner = Spinner(text='Выберите станок', size_hint_y=None, height=dp(44))
        self.layout.add_widget(self.machine_spinner)
        self.layout.add_widget(Label(text='Масло:', size_hint_y=None, height=dp(25)))
        self.oil_spinner = Spinner(text='Выберите масло', size_hint_y=None, height=dp(44))
        self.layout.add_widget(self.oil_spinner)
        self.layout.add_widget(Label(text='Литры:', size_hint_y=None, height=dp(25)))
        self.liters_input = TextInput(text='', hint_text='0.0', size_hint_y=None, height=dp(44), input_filter='float')
        self.layout.add_widget(self.liters_input)
        btn_save = Button(text='СОХРАНИТЬ', size_hint=(0.8, None), height=dp(50), pos_hint={'center_x': 0.5})
        btn_save.bind(on_press=self.save_record)
        self.layout.add_widget(btn_save)
        self.add_widget(self.layout)

    def on_enter(self):
        try:
            shops = get_shops()
            self.shop_map = {name: sid for sid, name in shops}
            shop_names = list(self.shop_map.keys())
            self.shop_spinner.values = shop_names if shop_names else ['Нет цехов']
            oils = get_oils()
            self.oil_map = {name: oid for oid, name in oils}
            oil_names = list(self.oil_map.keys())
            self.oil_spinner.values = oil_names if oil_names else ['Нет масел']
        except Exception as e:
            self.show_popup(f'Ошибка загрузки: {str(e)}')

    def on_shop_select(self, spinner, text):
        shop_id = self.shop_map.get(text)
        if shop_id:
            machines = get_machines(shop_id)
            self.machine_map = {f"{code} - {name}": mid for mid, code, name in machines}
            names = list(self.machine_map.keys())
            self.machine_spinner.values = names if names else ['Нет станков']
        else:
            self.machine_spinner.values = ['Нет станков']

    def save_record(self, instance):
        machine = self.machine_spinner.text
        oil = self.oil_spinner.text
        liters = self.liters_input.text
        date = self.date_input.text
        if not all([machine, oil, liters]) or 'Нет' in machine or 'Нет' in oil:
            self.show_popup('Ошибка', 'Заполните все поля')
            return
        try:
            liters = float(liters)
        except:
            self.show_popup('Ошибка', 'Литры - число')
            return
        machine_id = self.machine_map.get(machine)
        oil_id = self.oil_map.get(oil)
        if not machine_id or not oil_id:
            self.show_popup('Ошибка', 'Неверный выбор')
            return
        add_record(date, machine_id, oil_id, liters)
        self.liters_input.text = ''
        self.show_popup('Успех', 'Запись сохранена')

    def show_popup(self, title, text):
        Popup(title=title, content=Label(text=text), size_hint=(0.7, 0.3)).open()

class JournalScreen(Screen):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        layout = BoxLayout(orientation='vertical')
        header = BoxLayout(size_hint_y=None, height=dp(45))
        btn_back = Button(text='< Назад', size_hint_x=0.3)
        btn_back.bind(on_press=lambda x: setattr(self.manager, 'current', 'main'))
        header.add_widget(btn_back)
        header.add_widget(Label(text='Журнал записей', size_hint_x=0.7))
        layout.add_widget(header)
        self.scroll = ScrollView()
        self.list_layout = GridLayout(cols=1, spacing=dp(2), size_hint_y=None)
        self.list_layout.bind(minimum_height=self.list_layout.setter('height'))
        self.scroll.add_widget(self.list_layout)
        layout.add_widget(self.scroll)
        self.add_widget(layout)

    def on_enter(self):
        self.list_layout.clear_widgets()
        try:
            records = get_records()
            for rec_id, date, machine, oil, liters in records:
                item = Button(text=f"{date} | {machine} | {oil} | {liters} л", size_hint_y=None, height=dp(40), halign='left')
                item.bind(on_long_press=lambda x, rid=rec_id: self.confirm_delete(rid))
                self.list_layout.add_widget(item)
            if not records:
                self.list_layout.add_widget(Label(text='Нет записей', size_hint_y=None, height=dp(40)))
        except Exception as e:
            self.list_layout.add_widget(Label(text=f'Ошибка: {str(e)}', size_hint_y=None, height=dp(60)))

    def confirm_delete(self, rec_id):
        delete_record(rec_id)
        self.on_enter()

class ReportScreen(Screen):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        layout = BoxLayout(orientation='vertical', padding=dp(10), spacing=dp(8))
        header = BoxLayout(size_hint_y=None, height=dp(45))
        btn_back = Button(text='< Назад', size_hint_x=0.3)
        btn_back.bind(on_press=lambda x: setattr(self.manager, 'current', 'main'))
        header.add_widget(btn_back)
        header.add_widget(Label(text='Отчет', size_hint_x=0.7))
        layout.add_widget(header)

        self.start_date = TextInput(text=datetime.now().replace(day=1).strftime('%Y-%m-%d'), size_hint_y=None, height=dp(40))
        self.end_date = TextInput(text=datetime.now().strftime('%Y-%m-%d'), size_hint_y=None, height=dp(40))
        layout.add_widget(Label(text='Период С:', size_hint_y=None, height=dp(20)))
        layout.add_widget(self.start_date)
        layout.add_widget(Label(text='По:', size_hint_y=None, height=dp(20)))
        layout.add_widget(self.end_date)

        btn_build = Button(text='Сформировать', size_hint_y=None, height=dp(45))
        btn_build.bind(on_press=self.build_report)
        layout.add_widget(btn_build)

        self.table_area = BoxLayout(orientation='vertical', size_hint_y=None)
        self.table_area.bind(minimum_height=self.table_area.setter('height'))
        scroll = ScrollView()
        scroll.add_widget(self.table_area)
        layout.add_widget(scroll)

        btn_box = BoxLayout(size_hint_y=None, height=dp(45), spacing=dp(10))
        btn_export = Button(text='Сохранить CSV')
        btn_export.bind(on_press=self.export_csv)
        btn_share = Button(text='Поделиться')
        btn_share.bind(on_press=self.share_csv)
        btn_box.add_widget(btn_export)
        btn_box.add_widget(btn_share)
        layout.add_widget(btn_box)

        self.add_widget(layout)

    def build_report(self, instance):
        self.table_area.clear_widgets()
        start = self.start_date.text
        end = self.end_date.text
        data = get_report(start, end)
        if not data:
            self.table_area.add_widget(Label(text='Нет данных', size_hint_y=None, height=dp(40)))
            return
        machines = {}
        all_oils = set()
        for machine, oil, liters in data:
            machines.setdefault(machine, {})[oil] = liters
            all_oils.add(oil)
        oils_sorted = sorted(all_oils)
        self.report_machines = sorted(machines.keys())
        self.report_oils = oils_sorted
        self.report_data = machines

        cols = len(oils_sorted) + 1
        table = GridLayout(cols=cols, spacing=dp(2), size_hint_y=None)
        table.bind(minimum_height=table.setter('height'))

        header_label = Label(text='[b]Станок[/b]', markup=True, size_hint_y=None, height=dp(30), halign='left', valign='middle')
        header_label.bind(size=header_label.setter('text_size'))
        table.add_widget(header_label)
        for oil in oils_sorted:
            h = Label(text=f'[b]{oil}[/b]', markup=True, size_hint_y=None, height=dp(30), halign='center', valign='middle')
            h.bind(size=h.setter('text_size'))
            table.add_widget(h)

        for m in self.report_machines:
            lbl = Label(text=m, size_hint_y=None, height=dp(30), halign='left', valign='middle')
            lbl.bind(size=lbl.setter('text_size'))
            table.add_widget(lbl)
            for oil in oils_sorted:
                val = machines[m].get(oil, 0)
                lbl = Label(text=str(val), size_hint_y=None, height=dp(30), halign='center', valign='middle')
                lbl.bind(size=lbl.setter('text_size'))
                table.add_widget(lbl)

        self.table_area.add_widget(table)

    def _generate_csv_text(self):
        lines = ["Станок," + ",".join(self.report_oils)]
        for m in self.report_machines:
            row = [m] + [str(self.report_data[m].get(o, 0)) for o in self.report_oils]
            lines.append(",".join(row))
        return "\n".join(lines)

    def export_csv(self, instance):
        if not hasattr(self, 'report_data'):
            self.show_popup('Сначала сформируйте отчет')
            return
        csv_text = self._generate_csv_text()
        dl = os.path.join(os.environ.get('EXTERNAL_STORAGE', '/storage/emulated/0'), 'Download')
        os.makedirs(dl, exist_ok=True)
        path = os.path.join(dl, 'otchet_maslo.csv')
        try:
            with open(path, 'w') as f:
                f.write(csv_text)
            self.show_popup(f'Сохранено в:\n{path}')
        except Exception as e:
            self.show_popup(f'Ошибка сохранения: {str(e)}')

    def share_csv(self, instance):
        if not PLYER_AVAILABLE:
            self.show_popup('Функция отправки недоступна')
            return
        if not hasattr(self, 'report_data'):
            self.show_popup('Сначала сформируйте отчет')
            return
        csv_text = self._generate_csv_text()
        db_dir = os.path.join(os.environ.get('ANDROID_APP_PATH', os.path.expanduser('~')), '.oil_data')
        os.makedirs(db_dir, exist_ok=True)
        path = os.path.join(db_dir, 'otchet_maslo.csv')
        with open(path, 'w') as f:
            f.write(csv_text)
        try:
            share.file(path, 'text/csv')
        except Exception as e:
            self.show_popup(f'Ошибка отправки: {str(e)}')

    def show_popup(self, text):
        Popup(title='', content=Label(text=text), size_hint=(0.6, 0.25)).open()

class ReferencesScreen(Screen):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        layout = BoxLayout(orientation='vertical', padding=dp(10), spacing=dp(8))
        header = BoxLayout(size_hint_y=None, height=dp(45))
        btn_back = Button(text='< Назад', size_hint_x=0.3)
        btn_back.bind(on_press=lambda x: setattr(self.manager, 'current', 'main'))
        header.add_widget(btn_back)
        header.add_widget(Label(text='Справочники', size_hint_x=0.7))
        layout.add_widget(header)

        layout.add_widget(Label(text='Просмотр и редактирование:', size_hint_y=None, height=dp(30)))
        btn_shop_list = Button(text='Список цехов', size_hint_y=None, height=dp(45))
        btn_shop_list.bind(on_press=lambda x: setattr(self.manager, 'current', 'shop_list'))
        layout.add_widget(btn_shop_list)
        btn_machine_list = Button(text='Список станков', size_hint_y=None, height=dp(45))
        btn_machine_list.bind(on_press=lambda x: setattr(self.manager, 'current', 'machine_list'))
        layout.add_widget(btn_machine_list)
        btn_oil_list = Button(text='Список масел', size_hint_y=None, height=dp(45))
        btn_oil_list.bind(on_press=lambda x: setattr(self.manager, 'current', 'oil_list'))
        layout.add_widget(btn_oil_list)

        layout.add_widget(Label(text='Быстрое добавление:', size_hint_y=None, height=dp(30)))
        layout.add_widget(Label(text='Новый цех:', size_hint_y=None, height=dp(25)))
        self.shop_input = TextInput(hint_text='Название цеха', size_hint_y=None, height=dp(40))
        layout.add_widget(self.shop_input)
        btn_add_shop = Button(text='Добавить цех', size_hint_y=None, height=dp(40))
        btn_add_shop.bind(on_press=self.add_shop)
        layout.add_widget(btn_add_shop)

        layout.add_widget(Label(text='Новый станок:', size_hint_y=None, height=dp(25)))
        self.machine_code = TextInput(hint_text='Код станка', size_hint_y=None, height=dp(40))
        layout.add_widget(self.machine_code)
        self.machine_name = TextInput(hint_text='Название (необязательно)', size_hint_y=None, height=dp(40))
        layout.add_widget(self.machine_name)
        self.ref_shop_spinner = Spinner(text='Цех', size_hint_y=None, height=dp(44))
        layout.add_widget(self.ref_shop_spinner)
        btn_add_machine = Button(text='Добавить станок', size_hint_y=None, height=dp(40))
        btn_add_machine.bind(on_press=self.add_machine)
        layout.add_widget(btn_add_machine)

        layout.add_widget(Label(text='Новое масло:', size_hint_y=None, height=dp(25)))
        self.oil_input = TextInput(hint_text='Марка масла', size_hint_y=None, height=dp(40))
        layout.add_widget(self.oil_input)
        btn_add_oil = Button(text='Добавить масло', size_hint_y=None, height=dp(40))
        btn_add_oil.bind(on_press=self.add_oil)
        layout.add_widget(btn_add_oil)

        self.add_widget(layout)

    def on_enter(self):
        try:
            shops = get_shops()
            self.shop_map = {name: sid for sid, name in shops}
            shop_names = list(self.shop_map.keys())
            if shop_names:
                self.ref_shop_spinner.values = shop_names
                if self.ref_shop_spinner.text not in shop_names:
                    self.ref_shop_spinner.text = shop_names[0]
            else:
                self.ref_shop_spinner.values = ['Нет цехов']
                self.ref_shop_spinner.text = 'Нет цехов'
        except Exception as e:
            self.show_popup(f'Ошибка загрузки: {str(e)}')

    def add_shop(self, instance):
        name = self.shop_input.text.strip()
        if not name: return
        ok, msg = add_shop(name)
        self.show_popup(msg)
        if ok:
            self.shop_input.text = ''
            self.on_enter()

    def add_machine(self, instance):
        code = self.machine_code.text.strip()
        name = self.machine_name.text.strip() or code
        shop_name = self.ref_shop_spinner.text
        if not code or shop_name in ('Цех', 'Нет цехов', ''):
            self.show_popup('Код и цех обязательны')
            return
        shop_id = self.shop_map.get(shop_name)
        if not shop_id:
            self.show_popup('Цех не найден')
            return
        ok, msg = add_machine(code, name, shop_id)
        self.show_popup(msg)
        if ok:
            self.machine_code.text = ''
            self.machine_name.text = ''

    def add_oil(self, instance):
        name = self.oil_input.text.strip()
        if not name: return
        ok, msg = add_oil(name)
        self.show_popup(msg)
        if ok:
            self.oil_input.text = ''

    def show_popup(self, text):
        Popup(title='', content=Label(text=text), size_hint=(0.6, 0.25)).open()


class BaseListScreen(Screen):
    def build_layout(self, title):
        self.layout = BoxLayout(orientation='vertical')
        header = BoxLayout(size_hint_y=None, height=dp(45))
        btn_back = Button(text='< Назад', size_hint_x=0.3)
        btn_back.bind(on_press=lambda x: setattr(self.manager, 'current', 'references'))
        header.add_widget(btn_back)
        header.add_widget(Label(text=title, size_hint_x=0.7))
        self.layout.add_widget(header)
        self.scroll = ScrollView()
        self.list_layout = GridLayout(cols=1, spacing=dp(2), size_hint_y=None)
        self.list_layout.bind(minimum_height=self.list_layout.setter('height'))
        self.scroll.add_widget(self.list_layout)
        self.layout.add_widget(self.scroll)
        self.add_widget(self.layout)

class ShopListScreen(BaseListScreen):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.build_layout('Цеха')

    def on_enter(self):
        self.list_layout.clear_widgets()
        shops = get_shops()
        for shop_id, name in shops:
            btn = Button(text=name, size_hint_y=None, height=dp(44))
            btn.bind(on_press=lambda x, sid=shop_id, sname=name: self.edit_shop(sid, sname))
            btn.bind(on_long_press=lambda x, sid=shop_id: self.confirm_delete_shop(sid))
            self.list_layout.add_widget(btn)
        if not shops:
            self.list_layout.add_widget(Label(text='Цехов нет', size_hint_y=None, height=dp(44)))

    def edit_shop(self, shop_id, current_name):
        popup_layout = BoxLayout(orientation='vertical', padding=dp(10), spacing=dp(8))
        popup_layout.add_widget(Label(text='Новое название:'))
        name_input = TextInput(text=current_name, size_hint_y=None, height=dp(44))
        popup_layout.add_widget(name_input)
        btn_save = Button(text='Сохранить', size_hint_y=None, height=dp(44))
        popup = Popup(title='Редактировать цех', content=popup_layout, size_hint=(0.8, 0.4))
        btn_save.bind(on_press=lambda x: self.save_shop(shop_id, name_input.text, popup))
        popup_layout.add_widget(btn_save)
        popup.open()

    def save_shop(self, shop_id, new_name, popup):
        if not new_name.strip(): return
        ok, msg = update_shop(shop_id, new_name.strip())
        popup.dismiss()
        self.show_popup(msg)
        if ok: self.on_enter()

    def confirm_delete_shop(self, shop_id):
        def delete(instance):
            delete_shop(shop_id)
            self.on_enter()
            popup.dismiss()
        popup = Popup(title='Удалить цех?', size_hint=(0.7, 0.35))
        content = BoxLayout(orientation='vertical', spacing=dp(5))
        content.add_widget(Label(text='Вместе со всеми станками цеха!'))
        btns = BoxLayout(size_hint_y=None, height=dp(44))
        btns.add_widget(Button(text='Отмена', on_press=popup.dismiss))
        btns.add_widget(Button(text='Удалить', on_press=delete))
        content.add_widget(btns)
        popup.content = content
        popup.open()

    def show_popup(self, text):
        Popup(title='', content=Label(text=text), size_hint=(0.6, 0.25)).open()

class MachineListScreen(BaseListScreen):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.build_layout('Станки')

    def on_enter(self):
        self.list_layout.clear_widgets()
        machines = get_machines()
        for mid, code, name in machines:
            display = f"{code} - {name}" if name else code
            btn = Button(text=display, size_hint_y=None, height=dp(44))
            btn.bind(on_press=lambda x, m_id=mid, m_code=code, m_name=name: self.edit_machine(m_id, m_code, m_name))
            btn.bind(on_long_press=lambda x, m_id=mid: self.confirm_delete_machine(m_id))
            self.list_layout.add_widget(btn)
        if not machines:
            self.list_layout.add_widget(Label(text='Станков нет', size_hint_y=None, height=dp(44)))

    def edit_machine(self, machine_id, current_code, current_name):
        popup_layout = BoxLayout(orientation='vertical', padding=dp(10), spacing=dp(8))
        popup_layout.add_widget(Label(text='Код:'))
        code_input = TextInput(text=current_code, size_hint_y=None, height=dp(44))
        popup_layout.add_widget(code_input)
        popup_layout.add_widget(Label(text='Название:'))
        name_input = TextInput(text=current_name or '', size_hint_y=None, height=dp(44))
        popup_layout.add_widget(name_input)
        popup_layout.add_widget(Label(text='Цех:'))
        shop_spinner = Spinner(text='Выберите цех', size_hint_y=None, height=dp(44))
        shops = get_shops()
        shop_map = {name: sid for sid, name in shops}
        shop_names = list(shop_map.keys())
        shop_spinner.values = shop_names if shop_names else ['Нет цехов']
        conn = sqlite3.connect(get_db_path())
        c = conn.cursor()
        c.execute("SELECT s.name FROM machines m JOIN shops s ON m.shop_id=s.id WHERE m.id=?", (machine_id,))
        row = c.fetchone()
        conn.close()
        if row and row[0] in shop_names:
            shop_spinner.text = row[0]
        elif shop_names:
            shop_spinner.text = shop_names[0]
        popup_layout.add_widget(shop_spinner)
        btn_save = Button(text='Сохранить', size_hint_y=None, height=dp(44))
        popup = Popup(title='Редактировать станок', content=popup_layout, size_hint=(0.85, 0.55))
        btn_save.bind(on_press=lambda x: self.save_machine(machine_id, code_input.text, name_input.text, shop_spinner.text, shop_map, popup))
        popup_layout.add_widget(btn_save)
        popup.open()

    def save_machine(self, machine_id, code, name, shop_name, shop_map, popup):
        if not code.strip() or shop_name in ('Выберите цех', 'Нет цехов', ''): return
        shop_id = shop_map.get(shop_name)
        if not shop_id: return
        ok, msg = update_machine(machine_id, code.strip(), name.strip() or code.strip(), shop_id)
        popup.dismiss()
        self.show_popup(msg)
        if ok: self.on_enter()

    def confirm_delete_machine(self, machine_id):
        def delete(instance):
            delete_machine(machine_id)
            self.on_enter()
            popup.dismiss()
        popup = Popup(title='Удалить станок?', size_hint=(0.7, 0.35))
        content = BoxLayout(orientation='vertical', spacing=dp(5))
        content.add_widget(Label(text='Будут удалены все записи по этому станку!'))
        btns = BoxLayout(size_hint_y=None, height=dp(44))
        btns.add_widget(Button(text='Отмена', on_press=popup.dismiss))
        btns.add_widget(Button(text='Удалить', on_press=delete))
        content.add_widget(btns)
        popup.content = content
        popup.open()

    def show_popup(self, text):
        Popup(title='', content=Label(text=text), size_hint=(0.6, 0.25)).open()

class OilListScreen(BaseListScreen):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.build_layout('Масла')

    def on_enter(self):
        self.list_layout.clear_widgets()
        oils = get_oils()
        for oil_id, name in oils:
            btn = Button(text=name, size_hint_y=None, height=dp(44))
            btn.bind(on_press=lambda x, oid=oil_id, oname=name: self.edit_oil(oid, oname))
            btn.bind(on_long_press=lambda x, oid=oil_id: self.confirm_delete_oil(oid))
            self.list_layout.add_widget(btn)
        if not oils:
            self.list_layout.add_widget(Label(text='Масел нет', size_hint_y=None, height=dp(44)))

    def edit_oil(self, oil_id, current_name):
        popup_layout = BoxLayout(orientation='vertical', padding=dp(10), spacing=dp(8))
        popup_layout.add_widget(Label(text='Новое название:'))
        name_input = TextInput(text=current_name, size_hint_y=None, height=dp(44))
        popup_layout.add_widget(name_input)
        btn_save = Button(text='Сохранить', size_hint_y=None, height=dp(44))
        popup = Popup(title='Редактировать масло', content=popup_layout, size_hint=(0.8, 0.35))
        btn_save.bind(on_press=lambda x: self.save_oil(oil_id, name_input.text, popup))
        popup_layout.add_widget(btn_save)
        popup.open()

    def save_oil(self, oil_id, new_name, popup):
        if not new_name.strip(): return
        ok, msg = update_oil(oil_id, new_name.strip())
        popup.dismiss()
        self.show_popup(msg)
        if ok: self.on_enter()

    def confirm_delete_oil(self, oil_id):
        def delete(instance):
            delete_oil(oil_id)
            self.on_enter()
            popup.dismiss()
        popup = Popup(title='Удалить масло?', size_hint=(0.7, 0.35))
        content = BoxLayout(orientation='vertical', spacing=dp(5))
        content.add_widget(Label(text='Будут удалены все записи с этим маслом!'))
        btns = BoxLayout(size_hint_y=None, height=dp(44))
        btns.add_widget(Button(text='Отмена', on_press=popup.dismiss))
        btns.add_widget(Button(text='Удалить', on_press=delete))
        content.add_widget(btns)
        popup.content = content
        popup.open()

    def show_popup(self, text):
        Popup(title='', content=Label(text=text), size_hint=(0.6, 0.25)).open()


class OilApp(App):
    def build(self):
        self.title = 'Учет масла'
        ensure_db()
        sm = ScreenManager()
        sm.add_widget(MainScreen(name='main'))
        sm.add_widget(AddRecordScreen(name='add_record'))
        sm.add_widget(JournalScreen(name='journal'))
        sm.add_widget(ReportScreen(name='report'))
        sm.add_widget(ReferencesScreen(name='references'))
        sm.add_widget(ShopListScreen(name='shop_list'))
        sm.add_widget(MachineListScreen(name='machine_list'))
        sm.add_widget(OilListScreen(name='oil_list'))
        return sm

if __name__ == '__main__':
    OilApp().run()
'''

with open('main.py', 'w', encoding='utf-8') as f:
    f.write(main_py)

# 7. Запись buildozer.spec
spec = """[app]
title = Учет масла
package.name = oilapp
package.domain = org.example
source.dir = .
source.include_exts = py,png,jpg,kv,atlas
version = 1.0
requirements = python3,kivy,plyer
orientation = portrait
fullscreen = 0
android.permissions = WRITE_EXTERNAL_STORAGE,READ_EXTERNAL_STORAGE
android.api = 33
android.minapi = 21
android.ndk = 25c
android.sdk = 33

[buildozer]
log_level = 2
warn_on_root = 1
"""

with open('buildozer.spec', 'w') as f:
    f.write(spec)

print("✅ Ячейка 1 завершена. Запускайте Ячейку 2.")

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
autoconf is already the newest version (2.71-2).
libffi-dev is already the newest version (3.4

In [4]:
import subprocess
%cd /content/oil_app

process = subprocess.Popen(
    ['bash', '-c', 'yes | buildozer android debug 2>&1'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end='')
    if 'APK' in line and 'created' in line.lower():
        print("\n>>> СБОРКА УСПЕШНО ЗАВЕРШЕНА <<<")

process.wait()

from google.colab import files
import glob
apk_files = glob.glob('bin/*.apk')
if apk_files:
    print(f"Найден APK: {apk_files[0]}")
    files.download(apk_files[0])
else:
    print("APK не найден. Прокрутите логи выше.")

Выходные данные были обрезаны до нескольких последних строк (5000).
Compiling '/content/oil_app/.buildozer/android/platform/build-arm64-v8a_armeabi-v7a/build/other_builds/python3/arm64-v8a__ndk_target_21/python3/Lib/importlib/readers.py'...
Listing '/content/oil_app/.buildozer/android/platform/build-arm64-v8a_armeabi-v7a/build/other_builds/python3/arm64-v8a__ndk_target_21/python3/Lib/importlib/resources'...
Compiling '/content/oil_app/.buildozer/android/platform/build-arm64-v8a_armeabi-v7a/build/other_builds/python3/arm64-v8a__ndk_target_21/python3/Lib/importlib/resources/__init__.py'...
Compiling '/content/oil_app/.buildozer/android/platform/build-arm64-v8a_armeabi-v7a/build/other_builds/python3/arm64-v8a__ndk_target_21/python3/Lib/importlib/resources/_adapters.py'...
Compiling '/content/oil_app/.buildozer/android/platform/build-arm64-v8a_armeabi-v7a/build/other_builds/python3/arm64-v8a__ndk_target_21/python3/Lib/importlib/resources/_common.py'...
Compiling '/content/oil_app/.buildoze

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>